# NanoJev: 입력에서 확률 출력까지

이 노트북은 [NanoJev 공개 소스](https://github.com/TianyuCodings/NanoJev/tree/76fdfc9ecdca45a9bcef17991a07d3041a87685a)를 실제로 import해 `request → candidate path → backbone/decision head → probability → response`를 추적합니다. Jev 제품의 내부 구현에 대한 주장은 아닙니다.

- **기본 실습:** 표준 라이브러리만 사용. 모델 가중치나 GPU 없이 토큰 경로와 출력 계산을 확인합니다. 기본 출력용 logit은 *설명용 가상 값*입니다.
- **선택 실습:** 공개 checkpoint, 호환 PyTorch/Transformers 및 CUDA가 있을 때만 실제 모델을 실행합니다.
- 소스 기준 커밋: `76fdfc9ecdca45a9bcef17991a07d3041a87685a`.


In [ ]:
from pathlib import Path
import importlib.util, json, math, os, subprocess, tempfile

REV = "76fdfc9ecdca45a9bcef17991a07d3041a87685a"
candidates = [Path(os.environ["NANOJEV_REPO"])] if os.getenv("NANOJEV_REPO") else []
candidates += [Path.cwd().parent / "nanojev", Path(tempfile.gettempdir()) / "learning-jev-nanojev"]
repo = next((p.resolve() for p in candidates if (p / "scripts/predict_toy_decisions.py").is_file()), None)
if repo is None:
    repo = candidates[-1]
    subprocess.run(["git", "clone", "https://github.com/tianyucodings/nanojev.git", str(repo)], check=True)
if not os.getenv("NANOJEV_REPO"):
    current = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    if current != REV:
        if repo == candidates[-1]:
            subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REV], check=True)
        else:
            print(f"주의: 로컬 NanoJev 커밋 {current[:12]}은 분석 기준 {REV[:12]}과 다릅니다.")
print("NanoJev source:", repo)


## 1. 요청 계약

원본 `scripts/predict_toy_decisions.py`의 `validate_request`와 `prepare_examples`를 사용합니다. 로컬 형식에서 `boolean`은 TypeSafe API의 `noul`과 의미가 가깝지만 동일한 wire 형식은 아닙니다. TypeSafe의 [primitive 설명](https://docs.typesafe.ai/primitives)과 [NanoJev 계약 비교](https://github.com/TianyuCodings/NanoJev/blob/76fdfc9ecdca45a9bcef17991a07d3041a87685a/docs/TYPESAFE_CONTRACT.md)를 참고하세요.


In [ ]:
spec = importlib.util.spec_from_file_location("nanojev_predict", repo / "scripts/predict_toy_decisions.py")
predict_src = importlib.util.module_from_spec(spec)
spec.loader.exec_module(predict_src)

payload = {"states": [{"id": "ticket-1", "state": "A customer was charged twice. Refund approved, but not yet paid. Service works normally.", "questions": {
    "team": {"type": "choice", "instructions": "Which team should handle the main issue?", "criteria": {"billing": "Payments and refunds", "technical": "Service outages", "account": "Sign-in problems"}},
    "paid": {"type": "boolean", "instructions": "Has the refund actually been paid? Approval alone is insufficient."},
    "impact": {"type": "score", "instructions": "How much is service use affected?", "criteria": ["No use obstacle", "Minor feature blocked", "Core feature blocked", "Service fully unavailable"]}
}}]}
predict_src.validate_request(payload)
print(json.dumps(payload, indent=2))


## 2. 질문 → candidate path

실제 `prepare_examples`를 문자 tokenizer로 실행합니다. 문자 tokenizer는 원본의 [계약 테스트](https://github.com/TianyuCodings/NanoJev/blob/76fdfc9ecdca45a9bcef17991a07d3041a87685a/scripts/test_question_contract.py)와 같은 진단 도구이며 Qwen tokenizer는 아닙니다. 원본은 state와 question prefix를 각 candidate path에 반복합니다. `id`와 `qid`는 결과 연결용이고 토큰에 들어가지 않습니다.


In [ ]:
class CharacterTokenizer:
    eos_token_id = 0
    def encode(self, text, add_special_tokens=False):
        assert add_special_tokens is False
        return [ord(ch) + 1 for ch in text]
    def decode(self, ids):
        return "".join(chr(token - 1) for token in ids if token != self.eos_token_id)

tokenizer = CharacterTokenizer()
examples = predict_src.prepare_examples(payload, tokenizer, max_length=100_000)
for ex in examples:
    print(f"{ex['qid']} | type={ex['type']} | output labels={ex['candidate_ids']} | backbone paths={len(ex['leaf_tokens'])}")
    for i, leaf in enumerate(ex['leaf_tokens']):
        print(f"  path {i}: {tokenizer.decode(leaf)!r}")
assert [len(x['leaf_tokens']) for x in examples] == [3, 1, 4]


In [ ]:
# 전송용 ID를 바꿔도 모델 입력 토큰은 그대로인지 확인
renamed = json.loads(json.dumps(payload))
renamed["states"][0]["id"] = "another-id"
renamed["states"][0]["questions"]["routing"] = renamed["states"][0]["questions"].pop("team")
other = predict_src.prepare_examples(renamed, tokenizer, max_length=100_000)
assert examples[0]["leaf_tokens"] == next(x for x in other if x["qid"] == "routing")["leaf_tokens"]
print("ID 변경: 기존 질문의 leaf_tokens 동일")


## 3. path batch → decision logits

[실제 `DecisionModel.forward`](https://github.com/TianyuCodings/NanoJev/blob/76fdfc9ecdca45a9bcef17991a07d3041a87685a/scripts/train_toy_decisions.py)는 모든 질문의 path를 펼쳐 `[path 수, 최대 토큰 길이]` 텐서를 만들고 backbone을 **한 번** 호출합니다. 이번 요청은 `3 + 1 + 4 = 8`개 path입니다. 이는 state prefix KV cache를 한 번만 계산한다는 뜻이 아닙니다.

`last_hidden_state`의 마지막 유효 토큰 → LayerNorm → 공유 scalar head가 기본 logit을 만듭니다. `set_head='attention'`일 때 **Choice에만** 후보 사이 MultiheadAttention의 보정 logit이 추가됩니다. Boolean은 semantic path 하나의 logit `z`를 `[0, z]`로 확장합니다. Score는 각 level path의 logit을 그대로 사용합니다. `lm_head`의 A/B/C 토큰 logit을 읽지 않습니다.


In [ ]:
flat_paths = [leaf for ex in examples for leaf in ex["leaf_tokens"]]
width = max(map(len, flat_paths))
print({"questions": len(examples), "candidate_paths": len(flat_paths), "padded_backbone_input_shape": (len(flat_paths), width)})
print("각 질문 path 수:", {ex["qid"]: len(ex["leaf_tokens"]) for ex in examples})


## 4. logits → 확률 → 응답 (가상 logit)

아래 logit은 모델 예측이 아닙니다. 원본 `DecisionPredictor.predict`의 `softmax(logits / temperature)`와 `answer_from_probabilities`를 분리해서 관찰하기 위한 값입니다. Boolean은 `[0, z]` softmax이므로 `p_true = sigmoid(z)`입니다. Score는 **0부터 시작하는 level index**의 확률 가중 평균입니다.


In [ ]:
def softmax(values, temperature=1.0):
    scaled = [v / temperature for v in values]
    peak = max(scaled)
    exp_values = [math.exp(v - peak) for v in scaled]
    total = sum(exp_values)
    return [v / total for v in exp_values]

example_logits = {"team": [2.0, 0.0, -1.0], "paid": [0.0, -1.5], "impact": [2.0, 0.0, -1.0, -2.0]}
answers = {}
for ex in examples:
    probabilities = softmax(example_logits[ex["qid"]])
    answers[ex["qid"]] = predict_src.answer_from_probabilities(ex, probabilities)
print(json.dumps(answers, indent=2))
assert abs(answers["paid"]["p_true"] - 1/(1+math.exp(1.5))) < 1e-12
assert abs(answers["impact"]["score"] - sum(i*p for i,p in enumerate(answers["impact"]["probabilities"].values()))) < 1e-12


In [ ]:
# temperature는 분포의 날카로움을 바꾸지만, 이를 맞췄다고 calibration이 자동 검증되지는 않습니다.
for temperature in (0.5, 1.0, 2.0):
    print(temperature, dict(zip(examples[0]["candidate_ids"], [round(p, 4) for p in softmax(example_logits["team"], temperature)])))


## 5. 선택: 공개 checkpoint로 실제 추론

[README의 quick start](https://github.com/TianyuCodings/NanoJev#quick-start)에 따라 `unified-games-v1`의 `best.safetensors`, `config.json`, `tokenizer/`, `backbone_config/`를 내려받고 `NANOJEV_CHECKPOINT`를 설정하세요. 원본 predictor는 CUDA를 요구합니다. 체크포인트가 없으면 이 셀은 안전하게 건너뜁니다. 실제 모델 출력은 설명용 logit 결과와 무관합니다.


In [ ]:
checkpoint = os.getenv("NANOJEV_CHECKPOINT")
if checkpoint:
    # NanoJev requirements-toy.txt에 맞는 torch/transformers/safetensors 환경과 CUDA 필요
    engine = predict_src.DecisionPredictor(checkpoint, disable_native_triton=True)
    actual = engine.predict(payload)
    print(json.dumps({"execution": actual["execution"], "states": actual["states"]}, indent=2))
else:
    print("건너뜀: NANOJEV_CHECKPOINT가 설정되지 않았습니다.")


## 직접 해 볼 변경

1. `team`에 새 선택지를 넣고 path 수, 입력 텐서 첫 차원, 결과 라벨이 어떻게 바뀌는지 확인하세요.
2. 다른 질문을 추가한 뒤 기존 질문의 `leaf_tokens`가 바뀌는지 확인하세요. 같은 요청의 **질문 간** attention은 없습니다.
3. Score level의 순서를 바꿔 보세요. 설명 문자열과 출력 index의 관계를 확인하세요.
4. 실제 checkpoint 실행 시 `execution.forward_passes`, `candidate_paths`, `prefix_sharing`, `autoregressive_decode_steps`를 확인하세요.
5. 관측된 정답이 있는 별도 데이터로 NLL/Brier/ECE를 계산해 calibration을 검증하세요. 이 노트북의 가상 logit이나 단일 요청으로 calibration을 판단할 수 없습니다.
